# Restore Vivienne's photos — free GPU (v3, self-logging)

This version pins the libraries that break CodeFormer on today's Colab, writes a **log to your Google Drive** at every step, and saves the result to Drive.

**Steps:**
1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. Click **Allow** for Google Drive when asked.
4. Wait ~15–25 min. You'll get **`restored.zip` in My Drive**. A file **`restore_log.txt`** also appears in My Drive — that's the diagnostic Marc's assistant can read.

Keep the tab open and the laptop awake. Originals are never touched.

In [ ]:
# 1) GPU check
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime → Change runtime type → T4 GPU, then Run all.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2) Mount Drive + set up a log file that survives disconnects
from google.colab import drive
drive.mount('/content/drive')
import os, datetime
DRIVE = '/content/drive/MyDrive'
LOG = DRIVE + '/restore_log.txt'
def log(msg):
    line = f'{datetime.datetime.now():%H:%M:%S}  {msg}'
    print(line, flush=True)
    with open(LOG, 'a') as f:
        f.write(line + '\n')
open(LOG, 'w').close()
log('=== run start ===')
log('drive mounted ok')

In [ ]:
# 3) Everything else in one guarded block: install -> restore -> zip -> Drive.
#    Any failure is written (with traceback) to restore_log.txt in your Drive.
import subprocess, traceback, shutil, glob

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        log(f'CMD FAILED: {cmd}')
        log('STDERR tail: ' + (r.stderr or '')[-1500:])
        raise RuntimeError(f'command failed: {cmd}')
    return r.stdout

try:
    log('pinning numpy<2 + opencv (the common breakage)')
    sh('pip install -q "numpy<2" opencv-python-headless')

    log('cloning CodeFormer')
    os.chdir('/content')
    sh('rm -rf CodeFormer && git clone -q https://github.com/sczhou/CodeFormer.git')
    os.chdir('/content/CodeFormer')
    log('installing requirements')
    sh('pip install -q -r requirements.txt')
    sh('python basicsr/setup.py develop')

    log('patching torchvision functional_tensor import')
    sh("find / -name degradations.py -path '*basicsr*' -exec sed -i 's/functional_tensor/functional/g' {} + 2>/dev/null || true")

    log('downloading model weights')
    sh('python scripts/download_pretrained_models.py facelib')
    sh('python scripts/download_pretrained_models.py CodeFormer')

    log('pulling photos from the public repo')
    os.chdir('/content')
    sh('rm -rf family && git clone -q --depth 1 https://github.com/MarcBittner/family.git')
    shutil.rmtree('/content/inputs', ignore_errors=True); os.makedirs('/content/inputs')
    src = '/content/family/montage/full'
    n = 0
    for f in sorted(os.listdir(src)):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            shutil.copy(os.path.join(src, f), '/content/inputs/' + f); n += 1
    log(f'{n} photos staged')

    log('running restoration on GPU (this is the long part)')
    os.chdir('/content/CodeFormer')
    sh('python inference_codeformer.py -w 0.7 --face_upsample -i /content/inputs -o /content/results')
    log('restoration finished')

    log('renaming + resizing + zipping')
    os.chdir('/content')
    fin = '/content/results/final_results'
    out = '/content/restored'
    shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
    cnt = 0
    for f in sorted(os.listdir(fin)):
        if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        stem = os.path.splitext(f)[0]
        subprocess.run(['convert', os.path.join(fin, f), '-resize', '1400x1400>',
                        '-quality', '88', os.path.join(out, 'p' + stem + '.jpg')])
        cnt += 1
    log(f'{cnt} restored & resized')
    sh('rm -f /content/restored.zip && cd /content && zip -q -r restored.zip restored')
    shutil.copy('/content/restored.zip', DRIVE + '/restored.zip')
    mb = os.path.getsize(DRIVE + '/restored.zip') // (1024*1024)
    log(f'DONE — restored.zip ({mb} MB) saved to My Drive. {cnt} photos.')
except Exception:
    log('FATAL:\n' + traceback.format_exc())
    raise